# Local smoke test

Same `train()` call as `train_colab.ipynb`'s Train cell, with tiny params and a local checkpoint dir instead of Drive -- just to check the training loop and tqdm progress bars render correctly in a real notebook kernel (as opposed to a plain terminal). Not for real training. Safe to delete once you've confirmed things work.

In [ ]:
import re
import tempfile
from pathlib import Path

from hive_bot.engine.constants import BASE_PIECE_TYPES
from hive_bot.model.network import HiveNet
from hive_bot.training.train import train

CHECKPOINT_DIR = Path(tempfile.mkdtemp())
ENABLED_TYPES = BASE_PIECE_TYPES

RUNS_PER_CELL_EXECUTION = 2
GAMES_PER_ITERATION = 3
SIMULATIONS_PER_MOVE = 8
BATCH_SIZE = 16
BATCHES_PER_ITERATION = 5
LEARNING_RATE = 1e-3
MAX_PLIES = 40

# Small on purpose -- this is just checking the plumbing/rendering, not
# training a real model.
NETWORK_KWARGS = {"trunk_channels": 8, "num_blocks": 1, "embed_dim": 4, "global_hidden": 8}

print("checkpoint dir:", CHECKPOINT_DIR)

In [ ]:
def latest_checkpoint(checkpoint_dir: Path) -> Path | None:
    checkpoints = list(checkpoint_dir.glob("checkpoint_*.pt"))
    if not checkpoints:
        return None
    return max(
        checkpoints, key=lambda p: int(re.search(r"checkpoint_(\d+)\.pt", p.name).group(1))
    )


resume_from = latest_checkpoint(CHECKPOINT_DIR)
print("resuming from:", resume_from if resume_from else "(nothing yet -- starting fresh)")

In [ ]:
model = train(
    iterations=RUNS_PER_CELL_EXECUTION,
    games_per_iter=GAMES_PER_ITERATION,
    simulations=SIMULATIONS_PER_MOVE,
    batch_size=BATCH_SIZE,
    batches_per_iter=BATCHES_PER_ITERATION,
    lr=LEARNING_RATE,
    enabled_types=ENABLED_TYPES,
    checkpoint_dir=CHECKPOINT_DIR,
    resume_from=resume_from,
    model=HiveNet(**NETWORK_KWARGS),
    max_plies=MAX_PLIES,
)

resume_from = latest_checkpoint(CHECKPOINT_DIR)
print("checkpoints so far:", sorted(p.name for p in CHECKPOINT_DIR.glob("*.pt")))

Re-run the cell above as many times as you like -- it picks up from `resume_from` each time, same as the real notebook's Train cell.